# 01 - Data Exploration

This notebook looks at the Group 5 data before modeling. We focus on the three ACORN segments, their daily demand levels, the shape of a typical day, and how weather changes consumption.

The point here is to understand what the model needs to learn. If the same pattern appears in the plots, it should later become a feature or a validation check.


## Setup

We import the project loading functions and the plotting libraries. The notebook reads the processed group data created by the pipeline, so it uses the same inputs as the modeling scripts.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from group5_energy.config import ACORN_GROUPS
from group5_energy.pipeline import load_daily_history, load_half_hourly_history, load_daily_weather

sns.set_theme(style="whitegrid")
try:
    from IPython.display import display
except ImportError:
    display = None


def show_plot(fig):
    if display is not None:
        display(fig)
    plt.close(fig)


## Load the Group 5 data

First we load the daily consumption, half-hourly consumption, and daily weather tables. The printed shapes and date ranges are a quick check that we are studying the expected history before the forecast period.


In [ ]:
daily = load_daily_history()
half = load_half_hourly_history()
weather = load_daily_weather()

print("Daily rows:", daily.shape)
print("Half-hourly rows:", half.shape)
print("ACORN groups:", ACORN_GROUPS)
print("Daily date range:", daily["Date"].min(), "to", daily["Date"].max())
print("Half-hourly date range:", half["DateTime"].min(), "to", half["DateTime"].max())

The loaded tables cover the three Group 5 ACORN segments only. This matters because the assignment templates also expect only ACORN-E, ACORN-F, and ACORN-Q.

At this stage we are checking the modeling data, not changing the raw client files. The irregularity notebook explains the earlier cleaning choices in more detail.


## Consumption level by segment

Before looking at curves, we compare simple daily statistics by ACORN segment. This gives a useful baseline for interpreting all later plots.


In [ ]:
daily.groupby("Acorn")["Conso_kWh"].agg(["count", "mean", "min", "max"]).round(3)

ACORN-E has the highest average daily consumption, ACORN-F is in the middle, and ACORN-Q is the lowest. That ranking is stable enough that the models should not treat the three groups as interchangeable.

The minimum and maximum values also show that there is meaningful variation within each group. A constant segment average would be too weak for the forecasting task.


## Daily seasonality

The next plot smooths daily consumption with a 14-day rolling mean. Smoothing removes some day-to-day noise so we can see the broader seasonal movement.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for acorn, group in daily.sort_values("Date").groupby("Acorn"):
    rolling = group.set_index("Date")["Conso_kWh"].rolling(14, min_periods=1).mean()
    ax.plot(rolling.index, rolling.values, label=f"{acorn} - {ACORN_GROUPS[acorn]}")
ax.set_title("Daily electricity consumption, 14-day rolling mean")
ax.set_ylabel("kWh")
ax.legend()
show_plot(fig)

The rolling lines show a clear winter pattern. Consumption rises during colder periods and falls during warmer periods.

The three ACORN segments move in a similar direction over time, but at different levels. This supports using shared calendar and weather features while still keeping ACORN as an important feature.


## Shape of a typical day

For the short-term forecast, the half-hourly profile is very important. We average consumption by half-hour slot to see what a normal day looks like for each segment.


In [ ]:
profile = half.copy()
profile["half_hour_slot"] = profile["DateTime"].dt.hour * 2 + (profile["DateTime"].dt.minute // 30)
profile = profile.groupby(["Acorn", "half_hour_slot"], as_index=False)["Conso_moy"].mean()
profile["time_of_day"] = profile["half_hour_slot"] / 2

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=profile, x="time_of_day", y="Conso_moy", hue="Acorn", ax=ax)
ax.set_title("Typical half-hourly profile")
ax.set_xlabel("Hour of day")
show_plot(fig)

The average day has lower demand overnight and higher demand later in the day. The evening period is especially important because it creates a peak that a simple daily model would miss.

This is why the short-term model includes half-hour slot, hour, weekday, lag, and rolling features. The model needs to know both the time of day and the recent consumption level.


## Temperature relationship

The last check joins daily consumption to mean temperature and computes the correlation by ACORN segment. This is a simple way to test whether weather should be included in the model.


In [ ]:
daily_weather = daily.merge(weather[["Date", "temperatureMean"]], on="Date", how="left")
daily_weather.groupby("Acorn").apply(
    lambda group: group["Conso_kWh"].corr(group["temperatureMean"]),
    include_groups=False,
).rename("temperature_consumption_corr").round(3)

The correlations are strongly negative for all three segments. When the mean temperature is lower, electricity consumption is higher.

This does not prove causality by itself, but it is consistent with winter heating behavior and gives a clear reason to keep temperature features in both the daily and half-hourly models.
